<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_7_model_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_7_model_transformers

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Instalación de librerías


### 0.2. Importación de librerías


In [1]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.6.97+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.2
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [4]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [5]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [6]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [7]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [8]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [9]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [10]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [11]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [12]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [13]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [14]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [15]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [16]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [17]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [25]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [26]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [27]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [28]:
transformers_metrics, metrics = load_or_create_metrics("4_7_transformers_metrics")

Las métricas no existen. Se crea el dataset transformers_metrics para almacenar las métricas


### 3.2. Función para guardar métricas

In [29]:
def save_metrics (metrics,  metrics_name: str):
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [30]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [31]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## 4. Re-formateo más Encoder mínimo

Helper para re-formatear nuestras ventanas 2D a 3D que el formato que el modelo necesita.

### 4.1. Helper: de 2D (aplanado) a 3D (B, T, F)

Lo usamos para cada set y horizonte. Nuestro `window_size = 90` y los `n_features` depende del horizonte de tiempo. El siguiente código valida que `windows_size * n_features == X.shape[1]`

In [32]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

In [33]:
window_size = 90
features_base = ['open','high','close','low','volume']

In [34]:
n_features_30 = len (features_base + features_to_30)
n_features_60 = len (features_base + features_to_60)
n_features_90 = len (features_base + features_to_90)

In [35]:
# Re-shape de tus matrices 2D -> 3D
Xtr_30   =   reshape_windows(X_train_30_scaled, window_size, n_features_30)
Xva_30  =   reshape_windows(X_valid_30_scaled, window_size, n_features_30)
Xte_30  = reshape_windows(X_test_30_scaled, window_size, n_features_30)

Xtr_60   =   reshape_windows(X_train_60_scaled, window_size, n_features_60)
Xva_60  =   reshape_windows(X_valid_60_scaled, window_size, n_features_60)
Xte_60  = reshape_windows(X_test_60_scaled, window_size, n_features_60)

Xtr_90   =   reshape_windows(X_train_90_scaled, window_size, n_features_90)
Xva_90  =   reshape_windows(X_valid_90_scaled, window_size, n_features_90)
Xte_90  = reshape_windows(X_test_90_scaled, window_size, n_features_90)

# Comprobación de shapes

print('\nReshape de ventanas 30min:\n')
print(f'\tXtr_30.shape:\t{Xtr_30.shape}')
print(f'\tXva_30.shape:\t{Xva_30.shape}')
print(f'\tXte_30.shape:\t{Xte_30.shape}')

print('\nReshape de ventanas 60min:\n')
print(f'\tXtr_60.shape:\t{Xtr_60.shape}')
print(f'\tXva_60.shape:\t{Xva_60.shape}')
print(f'\tXte_60.shape:\t{Xte_60.shape}')

print('\nReshape de ventanas 90min:\n')
print(f'\tXtr_90.shape:\t{Xtr_90.shape}')
print(f'\tXva_90.shape:\t{Xva_90.shape}')
print(f'\tXte_90.shape:\t{Xte_90.shape}')



Reshape de ventanas 30min:

	Xtr_30.shape:	(193487, 90, 10)
	Xva_30.shape:	(41567, 90, 10)
	Xte_30.shape:	(41567, 90, 10)

Reshape de ventanas 60min:

	Xtr_60.shape:	(193487, 90, 12)
	Xva_60.shape:	(41567, 90, 12)
	Xte_60.shape:	(41567, 90, 12)

Reshape de ventanas 90min:

	Xtr_90.shape:	(193487, 90, 12)
	Xva_90.shape:	(41567, 90, 12)
	Xte_90.shape:	(41567, 90, 12)


### 4.2. Encoder: (backbone + posición + TransformerEncoder)

In [36]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T]

class TimeSeriesEncoder(nn.Module):
    """
    Proyección a d_model + PositionalEncoding + TransformerEncoder (sin cabeza).
    Devuelve embeddings por paso temporal: (B, T, d_model)
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        # init
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)    # (B, T, D)
        z = self.pos_encoder(z)   # (B, T, D)
        z = self.encoder(z)       # (B, T, D)
        z = self.dropout(z)       # (B, T, D)
        return z

In [37]:
# Pasada por el encoder
device = "cuda" if torch.cuda.is_available() else "cpu"

#Enconder por cada horizonte
enc_30 = TimeSeriesEncoder(input_dim=n_features_30, d_model=128, nhead=8, num_layers=2).to(device)
enc_60 = TimeSeriesEncoder(input_dim=n_features_60, d_model=128, nhead=8, num_layers=2).to(device)
enc_90 = TimeSeriesEncoder(input_dim=n_features_90, d_model=128, nhead=8, num_layers=2).to(device)

enc_30 = enc_30.to(device)
enc_60 = enc_60.to(device)
enc_90 = enc_90.to(device)

In [38]:
xb_xtr_30 = torch.tensor(Xtr_30[:64], dtype=torch.float32).to(device)  # batch chico
xb_xva_30 = torch.tensor(Xva_30[:64], dtype=torch.float32).to(device)  # batch chico
xb_xte_30 = torch.tensor(Xte_30[:64], dtype=torch.float32).to(device)  # batch chico

xb_xtr_60 = torch.tensor(Xtr_60[:64], dtype=torch.float32).to(device)  # batch chico
xb_xva_60 = torch.tensor(Xva_60[:64], dtype=torch.float32).to(device)  # batch chico
xb_xte_60 = torch.tensor(Xte_60[:64], dtype=torch.float32).to(device)  # batch chico

xb_xtr_90 = torch.tensor(Xtr_90[:64], dtype=torch.float32).to(device)  # batch chico
xb_xva_90 = torch.tensor(Xva_90[:64], dtype=torch.float32).to(device)  # batch chico
xb_xte_90 = torch.tensor(Xte_90[:64], dtype=torch.float32).to(device)  # batch chico

pairs = [
    (xb_xtr_30, enc_30, "30-train"),
    (xb_xva_30, enc_30, "30-valid"),
    (xb_xte_30, enc_30, "30-test"),
    (xb_xtr_60, enc_60, "60-train"),
    (xb_xva_60, enc_60, "60-valid"),
    (xb_xte_60, enc_60, "60-test"),
    (xb_xtr_90, enc_90, "90-train"),
    (xb_xva_90, enc_90, "90-valid"),
    (xb_xte_90, enc_90, "90-test"),
]

for xb, enc, tag in pairs:
    with torch.no_grad():
        z = enc(xb)
    print(tag, z.shape)

30-train torch.Size([64, 90, 128])
30-valid torch.Size([64, 90, 128])
30-test torch.Size([64, 90, 128])
60-train torch.Size([64, 90, 128])
60-valid torch.Size([64, 90, 128])
60-test torch.Size([64, 90, 128])
90-train torch.Size([64, 90, 128])
90-valid torch.Size([64, 90, 128])
90-test torch.Size([64, 90, 128])


## 5. Pooling (Sin cambiar enconder)

Tenemos dos opciones simples (no requieren modificar el encoder):

- `mean`: promedio temporal.
- `last`: último paso temporal.

In [39]:
import torch
import torch.nn as nn

class TemporalPooling(nn.Module):
    def __init__(self, mode: str = "mean"):
        super().__init__()
        assert mode in ("mean", "last"), "Soportado: 'mean' o 'last'"
        self.mode = mode

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, D)
        if self.mode == "mean":
            return z.mean(dim=1)      # (B, D)
        else:  # "last"
            return z[:, -1, :]        # (B, D)

In [40]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

for xb, enc, tag in pairs:
  with torch.no_grad():
      z = enc(xb)  # (64, 90, 128)
      p1 = pool_mean(z)      # (64, 128)
      p2 = pool_last(z)      # (64, 128)
  print(f'Para {tag}\n\tPool mean:\t{p1.shape}\tPool last:\t{p2.shape}')


Para 30-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 30-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 30-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 60-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 60-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 60-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 90-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 90-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para 90-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])


## 6. Cabeza de regresión (salida escalar)

Una cabeza chiquita y estándar:

In [41]:
class RegressionHead(nn.Module):
    def __init__(self, d_model: int = 128, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, D)
        return self.net(x).squeeze(-1)   # (B,)

Sanity check end-to-end (sin entrenar, solo shapes y un MSE “dummy”):

In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pool = TemporalPooling("mean").to(device)
head = RegressionHead(128, 0.1).to(device)

z_list = []
h_list = []
y_hat_list = []   #yhat: son las 64 predicciones del modelo (encoder→pooling→cabeza), forma (64,).

for xb, enc, tag in pairs:
  with torch.no_grad():
      z = enc(xb)     # (64, 90, 128)
      h = pool(z)               # (64, 128)
      yhat = head(h)            # (64,)
      z_list.append(z)
      h_list.append(h)
      y_hat_list.append(yhat)
  print(f'Para {tag}:\t{yhat.shape}')             # esperado: torch.Size([64])

Para 30-train:	torch.Size([64])
Para 30-valid:	torch.Size([64])
Para 30-test:	torch.Size([64])
Para 60-train:	torch.Size([64])
Para 60-valid:	torch.Size([64])
Para 60-test:	torch.Size([64])
Para 90-train:	torch.Size([64])
Para 90-valid:	torch.Size([64])
Para 90-test:	torch.Size([64])


In [43]:
# Para verificar pérdida rápida (suponiendo y_train_30 es np.ndarray):

# Se convierte los 64 targets reales del batch a un tensor en el mismo device:
yb_ytr_30 = torch.tensor(y_train_30[:64], dtype=torch.float32).to(device)
yb_yva_30 = torch.tensor(y_valid_30[:64], dtype=torch.float32).to(device)
yb_yte_30 = torch.tensor(y_test_30[:64], dtype=torch.float32).to(device)

yb_ytr_60 = torch.tensor(y_train_60[:64], dtype=torch.float32).to(device)
yb_yva_60 = torch.tensor(y_valid_60[:64], dtype=torch.float32).to(device)
yb_yte_60 = torch.tensor(y_test_60[:64], dtype=torch.float32).to(device)

yb_ytr_90 = torch.tensor(y_train_90[:64], dtype=torch.float32).to(device)
yb_yva_90 = torch.tensor(y_valid_90[:64], dtype=torch.float32).to(device)
yb_yte_90 = torch.tensor(y_test_90[:64], dtype=torch.float32).to(device)

pairs_y = [
    (yb_ytr_30, y_hat_list[0], "30-train"),
    (yb_yva_30, y_hat_list[1], "30-valid"),
    (yb_yte_30, y_hat_list[2], "30-test"),
    (yb_ytr_60, y_hat_list[3], "60-train"),
    (yb_yva_60, y_hat_list[4], "60-valid"),
    (yb_yte_60, y_hat_list[5], "60-test"),
    (yb_ytr_90, y_hat_list[6], "90-train"),
    (yb_yva_90, y_hat_list[7], "90-valid"),
    (yb_yte_90, y_hat_list[8], "90-test"),
]

for yb, yhat, tag in pairs_y:
    # Si yhat es numpy, conviértelo a tensor primero:
    if isinstance(yhat, np.ndarray):
        yhat = torch.from_numpy(yhat)
    # Asegurar 1D y mismo device/dtype
    yhat = yhat.to(device).float().view(-1)
    yb    = yb.to(device).float().view(-1)

    #Calcula el promedio del error cuadrático del batch:
    loss = torch.nn.functional.mse_loss(yhat, yb)
    print(f'Para {tag}:\t{tuple(yhat.shape)}\tMSE={float(loss):.6f}')

Para 30-train:	(64,)	MSE=0.006904
Para 30-valid:	(64,)	MSE=0.006766
Para 30-test:	(64,)	MSE=0.004056
Para 60-train:	(64,)	MSE=0.005293
Para 60-valid:	(64,)	MSE=0.007690
Para 60-test:	(64,)	MSE=0.007328
Para 90-train:	(64,)	MSE=0.051188
Para 90-valid:	(64,)	MSE=0.059898
Para 90-test:	(64,)	MSE=0.040205


Con esto validamos que el pipeline: encoder → pooling → cabeza produce un escalar por muestra y que todo coincide con `y`.

### Para evaluar todas las `y` (train/valid/test) en cada horizonte

Dos funcioneS:
1. `predict_set(...)`:

  Hace forward por lotes sobre un set completo (X_flat → reshape → encoder → pooling → cabeza) y devuelve y_pred como np.ndarray.

2. `compute_metrics(...)`:
  Calcula RMSE, MAE, R², SMAPE y Directional Accuracy.


Con `window_size` = 90:
  - Horizonte 30: total 900 → n_features_30=10
  - Horizontes 60 y 90: total 1080 → n_features_60 y 90 =12

In [44]:
import numpy as np
import torch

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    eps = 1e-12
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2) + eps
    r2   = float(1 - ss_res/ss_tot)
    smape = float(np.mean(2*np.abs(y_pred - y_true)/(np.abs(y_true)+np.abs(y_pred)+eps))*100)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}

@torch.no_grad()
def predict_set(enc, pool, head, X_flat: np.ndarray, window_size: int, n_features: int,
                device: str, batch_size: int = 4096) -> np.ndarray:
    # 1) reshape 2D -> 3D
    N, TF = X_flat.shape
    assert TF == window_size * n_features, f"Inconsistencia: {TF} != {window_size*n_features}"
    X = X_flat.reshape(N, window_size, n_features).astype(np.float32)

    # 2) forward en lotes
    enc.eval(); head.eval()
    preds = []
    for i in range(0, N, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        z  = enc(xb)               # (B, T, D)
        h  = pool(z)               # (B, D)
        yb = head(h).cpu().numpy() # (B,)
        preds.append(yb)
    return np.concatenate(preds, axis=0)

In [45]:
import numpy as np
import torch

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    eps = 1e-12
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2) + eps
    r2   = float(1 - ss_res/ss_tot)
    smape = float(np.mean(2*np.abs(y_pred - y_true)/(np.abs(y_true)+np.abs(y_pred)+eps))*100)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}

@torch.no_grad()
def predict_set(enc, pool, head, X_flat: np.ndarray, window_size: int, n_features: int,
                device: str, batch_size: int = 4096) -> np.ndarray:
    # 1) reshape 2D -> 3D
    N, TF = X_flat.shape
    assert TF == window_size * n_features, f"Inconsistencia: {TF} != {window_size*n_features}"
    X = X_flat.reshape(N, window_size, n_features).astype(np.float32)

    # 2) forward en lotes
    enc.eval(); head.eval()
    preds = []
    for i in range(0, N, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)
        z  = enc(xb)               # (B, T, D)
        h  = pool(z)               # (B, D)
        yb = head(h).cpu().numpy() # (B,)
        preds.append(yb)
    return np.concatenate(preds, axis=0)

Evaluamos train/valid/test para cada horizonte con lo que tenemos hasta el momento:
  - Encoders: enc_30, enc_60, enc_90 en device
  - Un pooling (p. ej. pool = TemporalPooling("mean").to(device))
  - Una cabeza: head = RegressionHead(128, 0.1).to(device)

Como aún no entrenamos el modelo, este paso solo servirá para probar el pipeline; los números no serán interpretados como 'buenos' o 'malos'.

In [46]:
T = 90   # window_size usado en tus secuencias
cfg = {
    "30": {
        "n_features": 10,
        "enc": enc_30,
        "Xtr": X_train_30_scaled, "ytr": y_train_30,
        "Xva": X_valid_30_scaled, "yva": y_valid_30,
        "Xte": X_test_30_scaled,  "yte": y_test_30,
    },
    "60": {
        "n_features": 12,
        "enc": enc_60,
        "Xtr": X_train_60_scaled, "ytr": y_train_60,
        "Xva": X_valid_60_scaled, "yva": y_valid_60,
        "Xte": X_test_60_scaled,  "yte": y_test_60,
    },
    "90": {
        "n_features": 12,
        "enc": enc_90,
        "Xtr": X_train_90_scaled, "ytr": y_train_90,
        "Xva": X_valid_90_scaled, "yva": y_valid_90,
        "Xte": X_test_90_scaled,  "yte": y_test_90,
    },
}

In [47]:
for h, d in cfg.items():
    ytr_pred = predict_set(d["enc"], pool, head, d["Xtr"], T, d["n_features"], device)
    yva_pred = predict_set(d["enc"], pool, head, d["Xva"], T, d["n_features"], device)
    yte_pred = predict_set(d["enc"], pool, head, d["Xte"], T, d["n_features"], device)
    print(f"H{h} train:", compute_metrics(d["ytr"], ytr_pred))
    print(f"H{h} valid:", compute_metrics(d["yva"], yva_pred))
    print(f"H{h} test :", compute_metrics(d["yte"], yte_pred))

H30 train: {'RMSE': 0.08391394730673156, 'MAE': 0.06517232608952608, 'R2': -923.1579541105102, 'SMAPE': 187.9819069907063, 'DirAcc': 0.5212236481003891}
H30 valid: {'RMSE': 0.08685266247993059, 'MAE': 0.06756919922452623, 'R2': -703.3779285874699, 'SMAPE': 188.21201173533112, 'DirAcc': 0.5134842543363726}
H30 test : {'RMSE': 0.086167357860015, 'MAE': 0.06764863403685524, 'R2': -1280.4774812372798, 'SMAPE': 188.89294122033502, 'DirAcc': 0.5184160511944571}
H60 train: {'RMSE': 0.12328725853756824, 'MAE': 0.08989122644389953, 'R2': -994.5816693433934, 'SMAPE': 185.36909848343439, 'DirAcc': 0.5183552383364257}
H60 valid: {'RMSE': 0.12425966716858347, 'MAE': 0.09094986891948711, 'R2': -715.1700745448136, 'SMAPE': 186.4820831100463, 'DirAcc': 0.5022974956094979}
H60 test : {'RMSE': 0.12671901226524276, 'MAE': 0.09432572882647947, 'R2': -1294.026024361706, 'SMAPE': 187.43837770449636, 'DirAcc': 0.49065364351528856}
H90 train: {'RMSE': 0.15355838659818977, 'MAE': 0.1263591434614923, 'R2': -104

Como resultado de esta etapa (sin entrenar), solo lo tomamos como sanity check de que:

- Todo corre sin errores.
- Las dimensiones y device están bien.
- El pipeline produce métricas sin romperse.

Después de entrenar (optimizar la cabeza junto con el encoder, o fine-tunear), esperamos que:
- MSE/RMSE baje comparado contra el baseline,
- R² suba
- SMAPE baje
- DirAcc por encima de 0.5 de forma consistente.

## 5. Entrenamiento

### 5.1. Dataset + DataLoader (reshape dentro)

In [48]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class WindowDataset(Dataset):
    def __init__(self, X_flat, y, T, F):
        assert X_flat.shape[1] == T*F
        self.X = X_flat.reshape(-1, T, F).astype(np.float32)
        self.y = y.astype(np.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.from_numpy(np.array(self.y[i]))

#Ejemplo de función solo para 30min
def make_loaders_30(Xtr, ytr, Xva, yva, T=90, F=10, bs=256):
    ds_tr = WindowDataset(Xtr, ytr, T, F)
    ds_va = WindowDataset(Xva, yva, T, F)
    return (DataLoader(ds_tr, batch_size=bs, shuffle=True, pin_memory=True),
            DataLoader(ds_va, batch_size=bs, shuffle=False, pin_memory=True))

#Función estandarizada
def make_loaders(Xtr, ytr, Xva, yva, T, F, bs=256, num_workers=2):
    """
    Crea DataLoaders para train y validación para cualquier horizonte temporal.

    Parámetros:
        Xtr, Xva : arrays 2D aplanados (N, T*F)
        ytr, yva : arrays 1D (N,)
        T        : window_size (número de pasos temporales)
        F        : n_features por paso temporal
        bs       : batch size
        num_workers : hilos para cargar datos
    """
    ds_tr = WindowDataset(Xtr, ytr, T, F)
    ds_va = WindowDataset(Xva, yva, T, F)
    dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True, pin_memory=True, num_workers=num_workers)
    dl_va = DataLoader(ds_va, batch_size=bs, shuffle=False, pin_memory=True, num_workers=num_workers)
    return dl_tr, dl_va


### 5.2. Modelo compacto (encoder + pooling mean + head)

In [49]:
import torch.nn as nn

pool = TemporalPooling("mean").to(device)
head = RegressionHead(128, 0.1).to(device)
model_30 = enc_30  # ya creado con input_dim=10, d_model=128
model_60 = enc_60
model_90 = enc_90

### 5.3. Loop de entrenamiento (MSE, AdamW, early stopping simple)

In [50]:
import torch
import numpy as np

dl_tr, dl_va = make_loaders_30(X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, T=90, F=10, bs=256)
opt = torch.optim.AdamW(list(model_30.parameters()) + list(head.parameters()), lr=3e-4, weight_decay=1e-4)
best_val = float("inf"); best = None; patience=8; noimp=0

scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))
for epoch in range(1, 51):
    # --- train ---
    model_30.train(); head.train()
    tr_loss = 0.0
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            z = model_30(xb)           # (B,T,128)
            h = pool(z)                # (B,128)
            yhat = head(h).view(-1)    # (B,)
            loss = torch.nn.functional.mse_loss(yhat, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(list(model_30.parameters())+list(head.parameters()), 1.0)
        scaler.step(opt); scaler.update()
        tr_loss += loss.item() * xb.size(0)
    tr_loss /= len(dl_tr.dataset)

    # --- valid ---
    model_30.eval(); head.eval()
    va_loss = 0.0
    with torch.no_grad():
        for xb, yb in dl_va:
            xb, yb = xb.to(device), yb.to(device)
            z = model_30(xb); h = pool(z); yhat = head(h).view(-1)
            va_loss += torch.nn.functional.mse_loss(yhat, yb).item() * xb.size(0)
    va_loss /= len(dl_va.dataset)

    print(f"Epoch {epoch:03d}  train={tr_loss:.6e}  valid={va_loss:.6e}")

    if va_loss < best_val - 1e-9:
        best_val = va_loss; noimp = 0
        best = ( {k: v.detach().cpu().clone() for k,v in model_30.state_dict().items()},
                 {k: v.detach().cpu().clone() for k,v in head.state_dict().items()} )
    else:
        noimp += 1
        if noimp >= patience:
            print("Early stopping."); break

# Cargar el mejor estado
if best is not None:
    model_30.load_state_dict(best[0]); head.load_state_dict(best[1])


Epoch 001  train=6.177515e-04  valid=1.704333e-05
Epoch 002  train=3.998503e-05  valid=1.185388e-05
Epoch 003  train=2.630358e-05  valid=1.088916e-05
Epoch 004  train=1.966748e-05  valid=9.274953e-06
Epoch 005  train=1.570516e-05  valid=9.348411e-06
Epoch 006  train=1.274986e-05  valid=8.679338e-06
Epoch 007  train=1.086434e-05  valid=8.445572e-06
Epoch 008  train=9.480491e-06  valid=8.969877e-06
Epoch 009  train=8.555262e-06  valid=8.830307e-06
Epoch 010  train=7.848370e-06  valid=8.096009e-06
Epoch 011  train=7.108431e-06  valid=7.759424e-06
Epoch 012  train=6.508750e-06  valid=7.847160e-06
Epoch 013  train=5.884909e-06  valid=7.778402e-06
Epoch 014  train=5.409925e-06  valid=8.040760e-06
Epoch 015  train=5.053024e-06  valid=7.893759e-06
Epoch 016  train=4.712555e-06  valid=7.858585e-06
Epoch 017  train=4.458290e-06  valid=7.912146e-06
Epoch 018  train=4.195922e-06  valid=7.941119e-06
Epoch 019  train=3.943892e-06  valid=8.287134e-06
Early stopping.


### 5.4. Métricas finales H=30 (train/valid/test)

In [53]:
ytr_pred = predict_set(model_30, pool, head, X_train_30_scaled, 90, 10, device)
yva_pred = predict_set(model_30, pool, head, X_valid_30_scaled, 90, 10, device)
yte_pred = predict_set(model_30, pool, head, X_test_30_scaled,  90, 10, device)

H30_train = compute_metrics(y_train_30, ytr_pred)
H30_valid = compute_metrics(y_valid_30, yva_pred)
H30_test = compute_metrics(y_test_30,  yte_pred)

print("H30 train:", H30_train)
print("H30 valid:", H30_valid)
print("H30 test :", H30_test)

H30 train: {'RMSE': 0.0022407881298257587, 'MAE': 0.0015662594285638033, 'R2': 0.34100898277906955, 'SMAPE': 117.14112784490429, 'DirAcc': 0.6912557432799102}
H30 valid: {'RMSE': 0.002785574319460987, 'MAE': 0.0016600746392130324, 'R2': 0.2754489897460797, 'SMAPE': 118.10757295569823, 'DirAcc': 0.6873721942887386}
H30 test : {'RMSE': 0.002044479879780951, 'MAE': 0.001484356500046856, 'R2': 0.2785751332288954, 'SMAPE': 117.071311262815, 'DirAcc': 0.690162869584045}


In [54]:
def weighted_avg_metrics(metrics_dicts, weights):
    keys = metrics_dicts[0].keys()
    total_w = sum(weights)
    avg = {}
    for k in keys:
        avg[k] = sum(m[k]*w for m,w in zip(metrics_dicts,weights)) / total_w
    return avg

# Si tus sets tienen estos tamaños:
n_train, n_valid, n_test = 193487, 41567, 41567
weights = [n_train, n_valid, n_test]
H30_avg_w = weighted_avg_metrics([H30_train, H30_valid, H30_test], weights)
print("H30 ponderado:", H30_avg_w)

H30 ponderado: {'RMSE': 0.0022931528545391934, 'MAE': 0.0015680494511267778, 'R2': 0.32177573564796097, 'SMAPE': 117.27586149872184, 'DirAcc': 0.6905079513124456}


In [57]:
H30_avg_w

{'RMSE': 0.0022931528545391934,
 'MAE': 0.0015680494511267778,
 'R2': 0.32177573564796097,
 'SMAPE': 117.27586149872184,
 'DirAcc': 0.6905079513124456}

In [58]:
transformers_metrics.loc["transformer_30_100%"] = [
    H30_avg_w["RMSE"],
    H30_avg_w["MAE"],
    H30_avg_w["R2"],
    H30_avg_w["SMAPE"],
    H30_avg_w["DirAcc"]
]


In [59]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
transformer_30_100%,0.002293,0.001568,0.321776,117.275861,0.690508


# Entrenamiento anterior

### 5.1. Entrenamiento 30min

#### 5.1.1. Con 30% de dataset

In [ ]:
mlp_30_30 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_30_subsampleado_30%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.3,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time 188s / 3,13min

Entrenando modelo MLP_30_subsampleado_30% (resample=30%)...
Métricas de MLP_30_subsampleado_30%:

	 RMSE:	 0.002537
	  MAE:	 0.001655
	   R2:	 0.333589
	SMAPE:	 120.453559
	DirAcc:	 0.671852


#### 5.1.2. Con 50% de dataset

In [ ]:
mlp_30_50 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_30_subsampleado_50%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Tiempo: 774s / 13min

El modelo fue entrenado anteriormente y las métricas ya fueron calculadas

Métricas de MLP_30_subsampleado_50%:

	 RMSE:	 0.002543
	  MAE:	 0.001607
	   R2:	 0.397152
	SMAPE:	 119.067749
	DirAcc:	 0.699514


#### 5.1.3. Con ventanas completas

In [ ]:
print("Omitimos este entrenamiento")
_ = '''
mlp_30_100 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_30_100%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=1.0,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Tiempo:
'''

Omitimos este entrenamiento


### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [ ]:
mlp_60_30 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_60_subsampleado_30%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.3,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time: 390s /6.5m

Entrenando modelo MLP_60_subsampleado_30% (resample=30%)...
Métricas de MLP_60_subsampleado_30%:

	 RMSE:	 0.003266
	  MAE:	 0.001943
	   R2:	 0.505579
	SMAPE:	 101.422228
	DirAcc:	 0.767442


#### 4.2.2. Con 50% de dataset

In [ ]:
mlp_60_50 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_60_subsampleado_50%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time: 484 seg

Entrenando modelo MLP_60_subsampleado_50% (resample=50%)...
Métricas de MLP_60_subsampleado_50%:

	 RMSE:	 0.003372
	  MAE:	 0.001955
	   R2:	 0.453291
	SMAPE:	 101.994321
	DirAcc:	 0.767887


#### 4.2.3. Con ventanas completas

In [ ]:
print("Omitimos este entrenamiento")
_ = '''
mlp_60_100 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_60_100%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=1.0,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time
'''

Omitimos este entrenamiento


### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [ ]:
mlp_90_30 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_90_subsampleado_30%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.3,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time: 4300.23 segundos.

Entrenando modelo MLP_90_subsampleado_30% (resample=30%)...
Métricas de MLP_90_subsampleado_30%:

	 RMSE:	 0.003607
	  MAE:	 0.002267
	   R2:	 0.619033
	SMAPE:	 93.339665
	DirAcc:	 0.794707


#### 4.3.2. Con 50% de dataset

In [ ]:
mlp_90_50 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_90_subsampleado_50%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time: 970 segundos

Entrenando modelo MLP_90_subsampleado_50% (resample=50%)...
Métricas de MLP_90_subsampleado_50%:

	 RMSE:	 0.004250
	  MAE:	 0.002273
	   R2:	 0.460338
	SMAPE:	 93.932838
	DirAcc:	 0.803060


#### 4.3.3. Con ventanas completas

In [ ]:
print("Omitimos este entrenamiento")
_ = '''
mlp_90_100 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_90_100%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=1.0,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time:
'''

Omitimos este entrenamiento


## 5. Recuperación de métricas

In [ ]:
mlp_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
MLP_30_subsampleado_30%,0.002537,0.001655,0.333589,120.453559,0.671852
MLP_30_subsampleado_50%,0.002543,0.001607,0.397152,119.067749,0.699514
MLP_60_subsampleado_30%,0.003266,0.001943,0.505579,101.422228,0.767442
MLP_60_subsampleado_50%,0.003372,0.001955,0.453291,101.994321,0.767887
MLP_90_subsampleado_30%,0.003607,0.002267,0.619033,93.339665,0.794707
MLP_90_subsampleado_50%,0.004250,0.002273,0.460338,93.932838,0.803060


In [ ]:
save_metrics(mlp_metrics, "4_5_mlp_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_5_mlp_metrics.parquet


Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.

In [ ]:
def generate_metrics_mlp():
  '''
  Ejecutar está función solo en caso de perder las métricas
  El objetivo es no volver a correr los entrenamientos
  '''
  mlp_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])

  mlp_30_30 = {
      "RMSE":	 0.002537,
      "MAE":	 0.001655,
      "R2":	 0.333589,
      "SMAPE":	 120.453559,
      "DirAcc":	 0.671852
  }

  mlp_30_50 = {
      "RMSE": 0.002543,
      "MAE": 0.001607,
      "R2": 0.397152,
      "SMAPE": 119.067749,
      "DirAcc": 0.699514
  }

  mlp_60_30 = {
      "RMSE": 0.003266,
      "MAE": 0.001943,
      "R2": 0.505579,
      "SMAPE": 101.422228,
      "DirAcc": 0.767442
  }

  mlp_60_50 = {
    'RMSE': 0.003372370312304722,
    'MAE': 0.001954590335922684,
    'R2': 0.4532909519346531,
    'SMAPE': 101.99432110111918,
    'DirAcc': 0.7678872155126786
 }

  mlp_90_30 = {
      'RMSE': 0.0036072614892137035,
      'MAE': 0.0022671873954739,
      'R2': 0.6190330484920656,
      'SMAPE': 93.33966545801738,
      'DirAcc': 0.7947072975140337
  }

  mlp_90_50 = {
    'RMSE': 0.00425000104407735,
    'MAE': 0.0022730454470971714,
    'R2': 0.4603383303081291,
    'SMAPE': 93.93283824100307,
    'DirAcc': 0.8030601934273204
      }

  mlp_metrics.loc['MLP_30_subsampleado_30%'] = mlp_30_30
  mlp_metrics.loc['MLP_30_subsampleado_50%'] = mlp_30_50

  mlp_metrics.loc['MLP_60_subsampleado_30%'] = mlp_60_30
  mlp_metrics.loc['MLP_60_subsampleado_50%'] = mlp_60_50

  mlp_metrics.loc['MLP_90_subsampleado_30%'] = mlp_90_30
  mlp_metrics.loc['MLP_90_subsampleado_50%'] = mlp_90_50


  save_metrics(mlp_metrics, "4_5_mlp_metrics")
  return mlp_metrics

In [ ]:
#Ejecutar esta función solo en caso de perder las métricas de entrenamiento
#mlp_metrics = generate_metrics_mlp()

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_3_lgbm_metrics.parquet


In [ ]:
mlp_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
MLP_30_subsampleado_30%,0.002537,0.001655,0.333589,120.453559,0.671852
MLP_30_subsampleado_50%,0.002543,0.001607,0.397152,119.067749,0.699514
MLP_60_subsampleado_30%,0.003266,0.001943,0.505579,101.422228,0.767442
MLP_60_subsampleado_50%,0.003372,0.001955,0.453291,101.994321,0.767887
MLP_90_subsampleado_30%,0.003607,0.002267,0.619033,93.339665,0.794707
